<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 10 · Input/Output Operations
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {"tables": "tables"}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

This chapter focuses on practical file-based workflows for financial data.


## Why Input/Output Matters in Finance


Financial workflows usually start by loading data from external systems and
end by writing reports or derived datasets back to storage.


## A Scratch Folder for Examples


Keep the chapter examples in a disposable scratch folder so you can experiment
without cluttering the project root.


In [ ]:
# Import `Path` so the examples build portable filesystem paths.
from pathlib import Path

In [ ]:
# Define a single root path for all files created in this chapter.
TMP_ROOT = Path("..") / "_tmp"
# Ensure that the `_tmp/` directory exists; if it is already present,
# nothing happens.
TMP_ROOT.mkdir(exist_ok=True)
# Inspect the absolute path so that you see where files are written on your
# machine.
TMP_ROOT.resolve()

## Basic File I/O with the Standard Library


The standard library is enough for plain text files and for serializing
Python objects when the format stays under your control.


### Writing and Reading Text Files


Many systems still exchange data through simple text files, so it helps to
know the basic read and write pattern.


In [ ]:
# Construct a path under `_tmp/` where the CSV file lives.
report_path = TMP_ROOT / "simple_report.csv"
header = "symbol,price,quantity,value\n"
# Prepare a small in-memory representation of a position report as text lines.
rows = [
    "AAPL,180.25,10,1802.50\n",
    "SPY,520.10,5,2600.50\n",
]
# Open the file with a context manager so it is always closed properly, even
# if an exception occurs.
with report_path.open("w", encoding="utf-8") as f:
    f.write(header)
    f.writelines(rows)

In [ ]:
# Read the complete file content back into memory as a single string.
report_path.read_text(encoding="utf-8")

### Serializing Python Objects with pickle


Pickle is convenient for short-lived artifacts that you fully control, but it
is not a safe interchange format.


In [ ]:
import pickle
# Prepare a nested dictionary that could represent a simple position book.
positions = {
    "AAPL": {"price": 180.25, "quantity": 10},
    "SPY": {"price": 520.10, "quantity": 5},
}
# Choose a binary file under `_tmp/` for the serialized object.
pickle_path = TMP_ROOT / "positions.pkl"
# Open the file in binary write mode so that `pickle.dump()` can write a
# byte stream.
with pickle_path.open("wb") as f:
    pickle.dump(positions, f)
# Read the byte stream back from disk and reconstruct the original object.
with pickle_path.open("rb") as f:
    loaded = pickle.load(f)
loaded == positions

## Storing Numerical Arrays with NumPy


NumPy offers compact binary formats for arrays, which are a better fit than
text when you repeatedly store intermediate results.


### Saving and Loading Arrays with np.save and np.load


The simplest NumPy I/O pattern is one array to `.npy` and back again
with `np.load()`.


In [ ]:
import numpy as np
# Create a reproducible random-number generator for all array examples.
rng = np.random.default_rng(seed=42)
# Simulate one year of daily returns as a `NumPy` array.
daily_returns = rng.normal(loc=0.0003, scale=0.01, size=252)
# Point to the binary `.npy` file that will hold the return series.
returns_path = TMP_ROOT / "daily_returns.npy"
# Write the array to disk in a compact binary `.npy` format.
np.save(returns_path, daily_returns)
# Load the array back into memory; the shape and dtype remain unchanged.
loaded_returns = np.load(returns_path)
np.allclose(daily_returns, loaded_returns)

### Bundling Multiple Arrays with np.savez


Use `.npz` when you want to store several related arrays together in one
archive.


In [ ]:
# Turn simulated returns into a simple price path that starts at 100.
prices = 100 * (1 + daily_returns).cumprod()
# Capture a small summary array with the minimum, maximum, and final value.
summary = np.array(
    [prices.min(), prices.max(), prices[-1]],
    dtype=float,
)
# Bundle the related arrays into one `.npz` archive.
npz_path = TMP_ROOT / "returns_and_prices.npz"
# Store several arrays in one compressed `.npz` file with named entries.
np.savez(
    npz_path,
    returns=daily_returns,
    prices=prices,
    summary=summary,
)
# Load the archive and inspect which named arrays it contains.
data = np.load(npz_path)
sorted(data.files)

## Input/Output with pandas


pandas sits at the center of many financial data workflows because it reads
and writes tabular data while preserving labels and indexes.


### Loading EOD Prices from CSV


Prefer a local CSV file and fall back to a remote URL if the local file is
missing.


In [ ]:
import pandas as pd
# Prefer the project-local CSV file when it is available.
LOCAL_EOD = Path("..") / "data" / "eod_data.csv"
# Fall back to the remote copy when the local file is missing.
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"
# Decide at runtime whether to load from the local CSV or from the remote
# fallback URL.
eod_source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

In [ ]:
# Parse the `Date` column as datetimes and use it as the index so that the
# result is a proper financial time series.
prices = pd.read_csv(
    eod_source,
    parse_dates=["Date"],
    index_col="Date",
)

In [ ]:
prices.head()

### Writing and Reading CSV Files


CSV remains the default exchange format, so it is worth knowing how to write
it and read it back with dates intact.


In [ ]:
# Persist the time series as a CSV file under `_tmp/`, including the date index.
csv_path = TMP_ROOT / "eod_prices.csv"
prices.to_csv(csv_path, index=True)
# Reload the CSV and restore the `DatetimeIndex` from the `Date` column.
reloaded = pd.read_csv(
    csv_path,
    parse_dates=["Date"],
    index_col="Date",
)
reloaded.equals(prices)

In [ ]:
# Process the CSV file in manageable chunks so you can compute aggregates or
# apply filters without loading everything at once.
chunk_iter = pd.read_csv(
    csv_path,
    chunksize=250,
    parse_dates=["Date"],
)
total_rows = 0
for chunk in chunk_iter:
    total_rows += len(chunk)
total_rows == len(prices)

### Working with JSON


JSON is useful when you want a readable interchange format for nested data or
small time-series windows.


In [ ]:
# Select a small window of the most recent rows for illustration.
latest = prices.iloc[-5:].copy()
json_path = TMP_ROOT / "latest_prices.json"
# Write the data to JSON using the `table` orientation, which stores both
# the schema and the values.
latest.to_json(json_path, orient="table", date_format="iso")
# Read the JSON file back into a `DataFrame`; `orient="table"` restores the
# index for you.
json_loaded = pd.read_json(json_path, orient="table")
json_loaded.equals(latest)

### Combining pandas with SQLite


SQLite is a compact relational database that works well when you want SQL
access and file-based storage together.


In [ ]:
import sqlite3
# Create the SQLite database file used for the SQL round-trip example.
db_path = TMP_ROOT / "eod_prices.sqlite"
# Open a database connection to a file under `_tmp/`; the file is created
# automatically if needed.
with sqlite3.connect(db_path) as con:
    # Store the entire `DataFrame` in a SQL table named `prices`,
    # replacing any previous version.
    prices.to_sql("prices", con=con, if_exists="replace")
    # Pull back a smaller subset with SQL so the database does the
    # filtering first.
    rows = pd.read_sql(
        "SELECT Date, AAPL, SPY FROM prices WHERE Date >= '2020-01-01'",
        con=con,
        parse_dates=["Date"],
    )
rows.head()

### Storing Time Series in HDF5


HDF5 is useful for large numerical datasets when you want key-based access and
compact storage.


In [ ]:
# Choose the HDF5 file used for the pandas round-trip example.
hdf_path = TMP_ROOT / "eod_prices.h5"
# Write the complete `DataFrame` to an HDF5 file under the key `prices`.
prices.to_hdf(hdf_path, key="prices", mode="w")
# Use an `HDFStore` context manager to read the data back into memory.
with pd.HDFStore(hdf_path, mode="r") as store:
    stored_prices = store["prices"]
stored_prices.equals(prices)

### Comparing SQLite and HDF5 for Larger Panels


A larger synthetic panel makes it easier to see how row-oriented SQLite access
differs from compressed HDF5 storage.


In [ ]:
n_rows = 100_000
columns = [f"no{i}" for i in range(5)]
rng = np.random.default_rng(seed=42)
values = rng.standard_normal((n_rows, len(columns)))
panel = pd.DataFrame(values, columns=columns)


In [ ]:
panel.head()

In [ ]:
# Choose the SQLite file used for the larger panel example.
sql_path = TMP_ROOT / "large_panel.sqlite"
# Store the panel in SQLite and let SQL apply the filter before
# materializing rows.
with sqlite3.connect(sql_path) as con:
    panel.to_sql("panel", con=con, if_exists="replace", index=False)
    subset_sql = pd.read_sql(
        "SELECT * FROM panel WHERE no1 > 1.0 AND no2 < -1.0",
        con=con,
    )

In [ ]:
subset_sql.head()

In [ ]:
# Choose the HDF5 file used for the compressed panel round-trip.
hdf_panel_path = TMP_ROOT / "large_panel.h5"
# Write the same panel to HDF5 with Blosc compression and read it back.
panel.to_hdf(
    hdf_panel_path,
    key="panel",
    mode="w",
    complevel=5,
    complib="blosc",
)
panel_hdf = pd.read_hdf(hdf_panel_path, key="panel")

In [ ]:
panel_hdf.equals(panel)

## High-Performance I/O with PyTables


PyTables exposes lower-level HDF5 features such as compressed tables,
extendable arrays, and chunked expressions.


### Creating Compressed Tables and Arrays


These examples use PyTables to store compressed tables and extendable arrays
for larger numerical workloads.


In [ ]:
# Import `PyTables` if it is available in the current environment.
try:
    import tables as tb
    HAS_PYTABLES = True
except ImportError:
    tb = None
    HAS_PYTABLES = False
    print("Install PyTables with `pip install tables` to run this section.")

In [ ]:
if HAS_PYTABLES:
    # Define the table layout with one integer date field and two float columns.
    class Returns(tb.IsDescription):
        date = tb.Int64Col(pos=0)
        no1 = tb.Float64Col(pos=1)
        no2 = tb.Float64Col(pos=2)
else:
    Returns = None

In [ ]:
if HAS_PYTABLES:
    pytables_path = TMP_ROOT / "pytables_large.h5"
    h5 = tb.open_file(pytables_path, mode="w")
    filters = tb.Filters(complevel=5, complib="blosc")
    table = h5.create_table(
        where="/",
        name="returns",
        description=Returns,
        filters=filters,
    )
    dates = np.arange(n_rows, dtype="int64")
    returns_block = np.column_stack(
        [panel["no1"].to_numpy(), panel["no2"].to_numpy()]
    )
    row = table.row
    for idx, (no1_val, no2_val) in enumerate(returns_block):
        row["date"] = int(dates[idx])
        row["no1"] = float(no1_val)
        row["no2"] = float(no2_val)
        row.append()
    table.flush()
else:
    print("Skipping PyTables table creation.")


In [ ]:
if HAS_PYTABLES:
    # Show the first few rows stored in the compressed PyTables table.
    print(table[:3])
else:
    print("No PyTables table to display.")

In [ ]:
if HAS_PYTABLES:
    # Create an extendable array that stores 252-value path vectors.
    ran_paths = h5.create_earray(
        where="/",
        name="paths",
        atom=tb.Float64Atom(),
        shape=(0, 252),
        filters=filters,
    )
    # Generate one reusable block of simulated paths.
    block = rng.standard_normal((1_000, 252))
    # Append the same block repeatedly to grow the on-disk array.
    for _ in range(100):
        ran_paths.append(block)
else:
    print("Skipping PyTables EArray creation.")

In [ ]:
if HAS_PYTABLES:
    # Inspect the extendable array handle and its current shape.
    print(ran_paths)
else:
    print("No PyTables EArray to display.")

In [ ]:
if HAS_PYTABLES:
    # Close the HDF5 file so all buffered data is written to disk.
    h5.close()

### Out-of-Memory Computations with EArray


PyTables can evaluate expressions directly against on-disk arrays so large
transforms do not need to materialize fully in memory.


In [ ]:
if HAS_PYTABLES:
    # Reopen the file in append mode to add a derived output array.
    h5 = tb.open_file(pytables_path, mode="a")
    paths = h5.root.paths
    # Allocate another extendable array with the same column shape.
    out = h5.create_earray(
        where="/",
        name="paths_transformed",
        atom=tb.Float64Atom(),
        shape=(0, paths.shape[1]),
        filters=paths.filters,
    )
    # Define the element-wise expression and stream the result straight to disk.
    expr = tb.Expr("3 * sin(x) + sqrt(abs(x))", uservars={"x": paths})
    expr.set_output(out)
    expr.eval()
else:
    print("Skipping PyTables out-of-memory expression.")

In [ ]:
if HAS_PYTABLES:
    # Compare the output shape to the input shape after the expression
    # evaluation.
    print(out.shape, paths.shape)
else:
    print("No transformed PyTables array to inspect.")

In [ ]:
if HAS_PYTABLES:
    # Close the file again after the out-of-memory expression finishes.
    h5.close()

## I/O with TsTables


`TsTables` builds on top of `PyTables` for append-once, query-many-times
time-series storage.


### Sample Data for TsTables


The chapter uses synthetic second-by-second data so time-range retrieval
is large enough to matter.


In [ ]:
import datetime as dt
# Use a dedicated random-number generator for the high-frequency simulation.
ts_rng = np.random.default_rng(seed=42)
# Simulate 10 days of one-second bars for three synthetic time series.
n_seconds = 10 * 24 * 60 * 60
n_series = 3
start = dt.datetime(2027, 1, 1)
index = pd.date_range(start, periods=n_seconds, freq="s")
# Convert one second into a fraction of a trading year.
dt_year = 1 / (252 * 6.5 * 60 * 60)
# Use a constant volatility in the geometric Brownian motion.
sigma = 0.2
# Draw shocks, compound them into log prices, and exponentiate back to levels.
shocks = ts_rng.standard_normal(size=(n_seconds, n_series))
increments = -0.5 * sigma**2 * dt_year + sigma * np.sqrt(dt_year) * shocks
log_paths = increments.cumsum(axis=0)
hf_prices = 100 * np.exp(log_paths)
cols = ["ts1", "ts2", "ts3"]
# Wrap the simulated prices in a timestamp-indexed `DataFrame`.
hf_df = pd.DataFrame(hf_prices, index=index, columns=cols)

In [ ]:
hf_df.info()

### Storing High-Frequency Data with TsTables


This notebook guards the example so the chapter still runs if `tstables` is
not installed. Install it with `pip install
git+https://github.com/yhilpisch/tstables.git`.


In [ ]:
if HAS_PYTABLES:
    # Import `tstables` so it registers the `create_ts()` extension with
    # PyTables.
    try:
        import tstables as tstab
        HAS_TSTABLES = True
    except ImportError:
        tstab = None
        HAS_TSTABLES = False
        print("Install TsTables to run this section.")
else:
    tstab = None
    HAS_TSTABLES = False
    print("PyTables is required before TsTables can be used.")

In [ ]:
if HAS_TSTABLES:
    # Describe the timestamp column first, followed by the three float series.
    class HFDesc(tb.IsDescription):
        timestamp = tb.Int64Col(pos=0)
        ts1 = tb.Float64Col(pos=1)
        ts2 = tb.Float64Col(pos=2)
        ts3 = tb.Float64Col(pos=3)
else:
    HFDesc = None

In [ ]:
if HAS_TSTABLES:
    # Create a fresh HDF5 file and a `TsTable` that matches the simulated data.
    ts_path = TMP_ROOT / "hf_ts.h5"
    h5_ts = tb.open_file(ts_path, mode="w")
    ts = h5_ts.create_ts("/", "ts", HFDesc)
    # Append the full intraday `DataFrame` in one call.
    ts.append(hf_df)
else:
    print("Skipping TsTables write example.")

In [ ]:
if HAS_TSTABLES:
    # Confirm that the created object is a `TsTable` instance.
    print(type(ts))
else:
    print("No TsTable object to inspect.")

### Retrieving Time Ranges Efficiently


The key capability is reading only the time window you need from a large
append-only store.


In [ ]:
if HAS_TSTABLES:
    # Obtain a time-series view that supports range-based retrieval.
    ts_series = h5_ts.root.ts._f_get_timeseries()
    # Read one three-day window from the 10-day sample.
    read_start = dt.datetime(2027, 1, 3, 0, 0)
    read_end = dt.datetime(2027, 1, 5, 23, 59)
    rows = ts_series.read_range(read_start, read_end)
else:
    rows = None
    print("Skipping TsTables range query.")

In [ ]:
if HAS_TSTABLES:
    # Inspect the schema and row count of the retrieved time-range slice.
    rows.info()
else:
    print("No TsTables result to inspect.")

In [ ]:
if HAS_TSTABLES:
    # Display the first few rows from the selected time window.
    print(rows.head())
else:
    print("No TsTables rows to display.")

In [ ]:
if HAS_TSTABLES:
    # Check the shape of the retrieved time-range slice.
    print(rows.shape)
else:
    print("No TsTables shape to display.")

In [ ]:
if HAS_TSTABLES:
    # Close the TsTables file handle after the range queries finish.
    h5_ts.close()

## Out-of-Core Patterns and Performance Preparation


The larger synthetic panel lets you compare full scans, filtered reads,
and storage-specific query paths without pretending that the tiny price
example is representative.


In [ ]:
# Write the large panel to plain CSV for sequential scan tests.
panel_csv_path = TMP_ROOT / "large_panel.csv"
panel.to_csv(panel_csv_path, index=False)
# Write the same panel to table-format HDF5 with indexed filter columns.
hdf_panel_table_path = TMP_ROOT / "large_panel_table.h5"
panel.to_hdf(
    hdf_panel_table_path,
    key="panel",
    mode="w",
    format="table",
    complevel=5,
    complib="blosc",
    data_columns=["no1", "no2"],
)

In [ ]:
panel_csv_path.stat().st_size, hdf_panel_table_path.stat().st_size

In [ ]:
# Track total rows, matched rows, and the running sum of `no3` across chunks.
chunk_rows = 0
filtered_rows = 0
filtered_sum = 0.0
for chunk in pd.read_csv(panel_csv_path, chunksize=100_000):
    # Apply the predicate inside each chunk before updating the aggregates.
    mask = (chunk["no1"] > 1.0) & (chunk["no2"] < -1.0)
    subset = chunk.loc[mask, ["no3"]]
    chunk_rows += len(chunk)
    filtered_rows += len(subset)
    filtered_sum += subset["no3"].sum()

In [ ]:
chunk_rows == len(panel), filtered_rows > 0

In [ ]:
filtered_sum / filtered_rows

In [ ]:
# Recreate the SQLite table so this filtered-read example is self-contained.
sql_path = TMP_ROOT / "large_panel.sqlite"
with sqlite3.connect(sql_path) as con:
    panel.to_sql("panel", con=con, if_exists="replace", index=False)
    # Let SQLite apply the predicate before pandas materializes the subset.
    subset_sql = pd.read_sql(
        "SELECT no1, no2, no3 FROM panel WHERE no1 > 1.0 AND no2 < -1.0",
        con=con,
    )
# Ask HDF5 for only the matching rows and only the required columns.
subset_hdf = pd.read_hdf(
    hdf_panel_table_path,
    key="panel",
    where="(no1 > 1.0) & (no2 < -1.0)",
    columns=["no1", "no2", "no3"],
)

In [ ]:
len(subset_sql), len(subset_hdf)

In [ ]:
np.allclose(subset_sql["no3"].mean(), subset_hdf["no3"].mean())

In [ ]:
%%timeit
# Time a full chunked CSV scan across the entire file.
chunk_rows = 0
for chunk in pd.read_csv(panel_csv_path, chunksize=100_000):
    chunk_rows += len(chunk)

In [ ]:
%%timeit
# Time the chunked CSV scan when the predicate is applied in Python.
filtered_rows = 0
for chunk in pd.read_csv(panel_csv_path, chunksize=100_000):
    mask = (chunk["no1"] > 1.0) & (chunk["no2"] < -1.0)
    filtered_rows += int(mask.sum())

In [ ]:
%%timeit
# Time the filtered SQLite read with the predicate pushed into SQL.
with sqlite3.connect(sql_path) as con:
    pd.read_sql(
        "SELECT no1, no2, no3 FROM panel WHERE no1 > 1.0 AND no2 < -1.0",
        con=con,
    )

In [ ]:
%%timeit
# Time the filtered HDF5 read with `where` and column pruning.
pd.read_hdf(
    hdf_panel_table_path,
    key="panel",
    where="(no1 > 1.0) & (no2 < -1.0)",
    columns=["no1", "no2", "no3"],
)

## Cleaning Up


In [ ]:
!ls $TMP_ROOT

In [ ]:
!rm $TMP_ROOT/*.csv
!rm $TMP_ROOT/*.h5
!rm $TMP_ROOT/*.json
!rm $TMP_ROOT/*.pkl
!rm $TMP_ROOT/*.npz
!rm $TMP_ROOT/*.sqlite
!rm $TMP_ROOT/*.npy

In [ ]:
!ls $TMP_ROOT

## Where We Are Heading Next


These patterns prepare you for the performance-focused techniques in the next
chapter.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
